# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
spaceship = pd.read_csv(url)

spaceship_clean = spaceship.dropna().copy()

spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]

spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])

spaceship_encoded = pd.get_dummies(
    spaceship_clean,
    columns=["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"],
    dtype=int
)

X = spaceship_encoded.drop(columns=["Transported"])
y = spaceship_encoded["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled training shape: (5284, 24)
Scaled test shape: (1322, 24)


In [4]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X_train.columns[selector.get_support()]

print("Selected features:")
print(selected_features.tolist())

Selected features:
['RoomService', 'Spa', 'VRDeck', 'HomePlanet_Earth', 'HomePlanet_Europa', 'CryoSleep_False', 'CryoSleep_True', 'Cabin_B', 'Destination_55 Cancri e', 'Destination_TRAPPIST-1e']


In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

knn_selected = KNeighborsClassifier()

knn_selected.fit(X_train_selected, y_train)

y_pred_selected = knn_selected.predict(X_test_selected)

print("Accuracy:", round(accuracy_score(y_test, y_pred_selected) * 100, 2), "%")
print(classification_report(y_test, y_pred_selected))

Accuracy: 76.48 %
              precision    recall  f1-score   support

       False       0.81      0.69      0.74       656
        True       0.73      0.84      0.78       666

    accuracy                           0.76      1322
   macro avg       0.77      0.76      0.76      1322
weighted avg       0.77      0.76      0.76      1322



**Perform Train Test Split**

In [6]:
from sklearn.model_selection import train_test_split

X = spaceship_encoded.drop(columns=["Transported"])
y = spaceship_encoded["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (5284, 24)
Test set: (1322, 24)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [9]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_train, y_train)
bagging_pred = bagging_model.predict(X_test)

print("Bagging accuracy:",
      round(accuracy_score(y_test, bagging_pred) * 100, 2), "%")

Bagging accuracy: 78.37 %


- Random Forests

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest accuracy:",
      round(accuracy_score(y_test, rf_pred) * 100, 2), "%")

Random Forest accuracy: 77.61 %


- Gradient Boosting

In [ ]:
#your code here

- Adaptive Boosting

In [ ]:
#your code here

Which model is the best and why?

In [11]:
results = pd.DataFrame({
    "Model": ["Bagging", "Pasting", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, bagging_pred),
        accuracy_score(y_test, pasting_pred),
        accuracy_score(y_test, rf_pred)
    ]
})

results["Accuracy (%)"] = (results["Accuracy"] * 100).round(2)

results[["Model", "Accuracy (%)"]].sort_values(
    "Accuracy (%)", ascending=False
)

NameError: name 'pasting_pred' is not defined